# Feast Feature Store — End-to-End Example

This notebook walks through the full Feast workflow on prokube:

1. Generate sample feature data
2. Configure the Feast client
3. Register features in the registry (`feast apply`)
4. Retrieve historical features for training
5. Train a model and log to MLflow
6. Materialize features to the Redis online store
7. Serve features online for inference

## Prerequisites

- A `FeatureStore` CR is deployed in your namespace (see `feast-cr.yaml`)
- The `feast-redis-config` secret exists in your namespace (ask your admin)


In [ ]:
!pip install -q feast scikit-learn mlflow

## 1. Generate sample data

We create a parquet file simulating hourly driver statistics over the past 7 days.
In a real scenario this would come from your data pipeline.


In [ ]:
import datetime
import os

import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000
now = datetime.datetime.now()
timestamps = [now - datetime.timedelta(hours=i) for i in range(n)]

driver_df = pd.DataFrame({
    "driver_id": np.random.choice([1001, 1002, 1003, 1004, 1005], n),
    "event_timestamp": timestamps,
    "conv_rate": np.random.uniform(0.1, 1.0, n).astype(np.float32),
    "acc_rate": np.random.uniform(0.5, 1.0, n).astype(np.float32),
    "avg_daily_trips": np.random.randint(1, 50, n).astype(np.int64),
    "created": timestamps,
})

os.makedirs("data", exist_ok=True)
driver_df.to_parquet("data/driver_stats.parquet")
print(f"Created {n} rows for {driver_df['driver_id'].nunique()} drivers")
driver_df.head()


## 2. Configure the Feast client

We build `feature_store.yaml` by reading the Redis connection string from the
`feast-redis-config` secret in our namespace.

The registry uses a local SQLite file (`/tmp/registry.db`). This is ephemeral
within the notebook session — re-run `feast apply` at the start of each session
to repopulate it. The Redis online store is persistent across sessions.


In [ ]:
import base64
import subprocess
import yaml


def get_namespace():
    """Read the current namespace from the pod's service account."""
    try:
        with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
            return f.read().strip()
    except FileNotFoundError:
        return subprocess.check_output(
            ["kubectl", "config", "view", "--minify", "-o", "jsonpath={..namespace}"]
        ).decode().strip()


def get_redis_connection_string():
    """Read the Redis connection string from the feast-redis-config secret.

    The secret has key 'redis' whose value is a YAML map:
        connection_string: "host:port,password=..."
    """
    result = subprocess.run(
        ["kubectl", "get", "secret", "feast-redis-config",
         "-o", "jsonpath={.data.redis}"],
        capture_output=True, text=True, check=True,
    )
    raw = base64.b64decode(result.stdout).decode()
    return yaml.safe_load(raw)["connection_string"]


NAMESPACE = get_namespace()
REDIS_CONNECTION_STRING = get_redis_connection_string()
FEAST_PROJECT = "my_features"  # must match spec.feastProject in your FeatureStore CR

feature_store_yaml = (
    f"project: {FEAST_PROJECT}\n"
    "provider: local\n"
    "offline_store:\n"
    "    type: file\n"
    "online_store:\n"
    "    type: redis\n"
    f"    connection_string: \"{REDIS_CONNECTION_STRING}\"\n"
    "registry:\n"
    "    registry_type: file\n"
    "    path: /tmp/registry.db\n"
    "auth:\n"
    "    type: no_auth\n"
    "entity_key_serialization_version: 3\n"
)

with open("feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written")
print(f"Namespace: {NAMESPACE}")
# Print host:port only — hide password
print(f"Redis: {REDIS_CONNECTION_STRING.split(',')[0]}")


## 3. Register features

`feast apply` reads `features.py` and writes the entity, data source, and
feature view definitions to the local SQLite registry (`/tmp/registry.db`).


In [ ]:
!feast apply

In [ ]:
# Verify what was registered
!feast feature-views list

## 4. Retrieve historical features for training

`get_historical_features` performs a **point-in-time join**: for each entity row,
it finds the most recent feature values as of that entity's timestamp. This
prevents data leakage — you only see features that were available when the
event occurred.


In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=".")

entity_df = pd.DataFrame({
    "driver_id": [1001, 1002, 1003, 1004, 1005],
    "event_timestamp": [now] * 5,
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df()

print("Training data (point-in-time correct):")
training_df


## 5. Train a model and log to MLflow

Use the Feast-provided training data to train a simple model, then log
everything to MLflow — including which Feast features were used.


In [ ]:
import mlflow
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Adjust the tracking URI if your cluster uses a different location
mlflow.set_tracking_uri("http://mlflow-mlflow-tracking-server.mlflow.svc.cluster.local:80")
mlflow.set_experiment("feast-driver-prediction")

FEATURE_COLS = ["acc_rate", "avg_daily_trips"]
TARGET = "conv_rate"

X = training_df[FEATURE_COLS].fillna(0)
y = training_df[TARGET].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="feast-driver-conv-rate") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_param("feast_project", FEAST_PROJECT)
    mlflow.log_param("feast_feature_view", "driver_hourly_stats")
    mlflow.log_param("features", ", ".join(FEATURE_COLS))
    mlflow.log_param("target", TARGET)
    mlflow.log_param("n_training_samples", len(X_train))
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(model, "driver_conv_model")

    print(f"MLflow run: {run.info.run_id}")
    print(f"MSE: {mse:.4f}, R2: {r2:.4f}")


## 6. Materialize features to Redis

Materialization reads the latest feature values from the offline parquet store
and writes them to Redis for low-latency online serving.

In production you would run this on a schedule (e.g. hourly cron job).


In [ ]:
!feast materialize-incremental $(date -u +'%Y-%m-%dT%H:%M:%S')

## 7. Online feature serving

Retrieve the latest feature values for specific entities. The Feast SDK reads
directly from Redis — no round-trip through the Feast server pod is needed.

This is what you call at inference time: given a `driver_id`, get their
current features to feed into the model.


In [ ]:
online_features = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}],
).to_dict()

print("Online features (latest values from Redis):")
for k, v in online_features.items():
    print(f"  {k}: {v}")


In [ ]:
# Use online features for inference
inference_df = pd.DataFrame(online_features)
predictions = model.predict(inference_df[FEATURE_COLS])

for driver_id, pred in zip(inference_df["driver_id"], predictions):
    print(f"Driver {driver_id}: predicted conv_rate = {pred:.4f}")


## Summary

| Step | Command / API | Purpose |
|------|--------------|--------|
| Define features | `features.py` | Declare entities, sources, feature views |
| Register | `feast apply` | Write definitions to local SQLite registry |
| Training data | `store.get_historical_features()` | Point-in-time correct join from parquet |
| Materialize | `feast materialize-incremental` | Push latest values to Redis online store |
| Online serving | `store.get_online_features()` | Sub-ms lookup from Redis by entity key |

### When to use Feast vs raw parquet

- **Use Feast** when you need consistent feature definitions across training and
  serving, point-in-time correctness, or low-latency online feature serving.
- **Use raw parquet** for one-off experiments where feature management overhead
  is not worth it.

### Note on the SQLite registry

This notebook uses `/tmp/registry.db` as the registry, which is local to the
notebook session. Re-run the "Configure" and `feast apply` cells at the start
of each new session. The Redis online store is persistent across sessions —
features materialized in one session are still available in subsequent ones.
